In [1]:
from keras.datasets import mnist
from keras.utils import to_categorical

In [2]:
#Showing directories’ size
import os, shutil

train_dir ='./images/train'
validation_dir = './images/val'
test_dir = './images/test'

train_corona_virus_d = './images/train/Corona_Virus_Disease'
train_normal = './images/train/Normal'
train_tuberculosis= './images/train/Tuberculosis'

val_corona_virus_d = './images/val/Corona_Virus_Disease'
val_normal = './images/val/Normal'
val_tuberculosis= './images/val/Tuberculosis'

test_corona_virus_d = './images/test/Corona_Virus_Disease'
test_normal = './images/test/Normal'
test_tuberculosis= './images/test/Tuberculosis'


print('total train corona virus images:', len(os.listdir(train_corona_virus_d)))
print('total train normal images:', len(os.listdir(train_normal)))
print('total train tuberculosis images:', len(os.listdir(train_tuberculosis)))


print('total validation corona virus images:', len(os.listdir(val_corona_virus_d)))
print('total validation normal images:', len(os.listdir(val_normal)))
print('total validation tuberculosis images:', len(os.listdir(val_tuberculosis)))


print('total testing organic corona virus img:', len(os.listdir(test_corona_virus_d)))
print('total testing recicle normal imgs:', len(os.listdir(test_normal)))
print('total testing recicle tuberculosis img:', len(os.listdir(test_tuberculosis)))

total train corona virus images: 1218
total train normal images: 1207
total train tuberculosis images: 1220
total validation corona virus images: 406
total validation normal images: 402
total validation tuberculosis images: 406
total testing organic corona virus img: 407
total testing recicle normal imgs: 404
total testing recicle tuberculosis img: 408


In [3]:
from keras.utils import image_dataset_from_directory

IMG_SIZE = 150
train_dataset = image_dataset_from_directory(
    train_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=32,
    label_mode='categorical',
    seed = 100
)

validation_dataset = image_dataset_from_directory(
    validation_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=64,
    label_mode='categorical',
    seed = 100
)

test_dataset = image_dataset_from_directory(
    test_dir,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=64,
    label_mode='categorical',
    seed = 100
)

Found 3645 files belonging to 3 classes.
Found 1214 files belonging to 3 classes.
Found 1219 files belonging to 3 classes.


In [8]:
from keras.applications.vgg16 import VGG16

# Crear el modelo base de VGG16 preentrenado en ImageNet
conv_base = VGG16(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))

In [9]:
# Congelar las capas de la base convolucional para que no se entrenen
conv_base.trainable = False

In [10]:
import tensorflow as tf
from tensorflow import keras
from keras import layers
import numpy as np

def objective(trial):
    opt_num_hidden_dense_units = trial.suggest_int("opt_num_hidden_dense_units", 10, 100)
    opt_lr = trial.suggest_float("opt_lr", 1e-6, 1e-2, log=True)
    opt_bs = trial.suggest_int("opt_bs", 16, 128)
    data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.1),
        layers.RandomZoom(0.2),
    ]
    )
    # Construir el modelo
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = data_augmentation(inputs)
    x = conv_base(x, training=False)
    x = layers.Flatten()(x)
    x = layers.Dense(opt_num_hidden_dense_units, activation="relu")(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(3, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    # Compilar el modelo
    model.compile(
        loss='categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=opt_lr),
        metrics=['acc'])
    history = model.fit(
        train_dataset,
        epochs=30,
        validation_data=validation_dataset,
        verbose=1,
        batch_size=opt_bs)
    min_val_loss = np.amin(history.history["val_loss"])
    return min_val_loss
    



In [11]:
import optuna as opt
study = opt.create_study()
study.optimize(objective, n_trials=5)


[I 2024-07-05 00:41:53,417] A new study created in memory with name: no-name-97a21864-8dd8-4de4-9986-2675e27779ae


Epoch 1/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - acc: 0.3675 - loss: 1.5768 - val_acc: 0.5873 - val_loss: 0.8304
Epoch 2/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 126s 1s/step - acc: 0.4928 - loss: 0.9675 - val_acc: 0.8138 - val_loss: 0.5946
Epoch 3/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 127s 1s/step - acc: 0.5813 - loss: 0.9002 - val_acc: 0.8171 - val_loss: 0.6458
Epoch 4/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 125s 1s/step - acc: 0.6029 - loss: 0.8394 - val_acc: 0.8015 - val_loss: 0.6360
Epoch 5/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 145s 1s/step - acc: 0.6199 - loss: 0.8141 - val_acc: 0.8616 - val_loss: 0.5775
Epoch 6/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 159s 1s/step - acc: 0.6351 - loss: 0.8403 - val_acc: 0.9069 - val_loss: 0.4477
Epoch 7/30
114/114 ━━━━━━━━━━━━━━━━━━━━ 134s 1s/step - acc: 0.6280 - loss: 0.8140 - val_acc: 0.8863 - val_loss: 0.4263
Epoch 8/30
 68/114 ━━━━━━━━━━━━━━━━━━━━ 40s 881ms/step - acc: 0.6570 - loss: 0.7177

[W 2024-07-05 00:58:35,197] Trial 0 failed with parameters: {'opt_num_hidden_dense_units': 12, 'opt_lr': 0.0011221379988906838, 'opt_bs': 84} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\alfre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\optuna\study\_optimize.py", line 196, in _run_trial
    value_or_values = func(trial)
  File "C:\Users\alfre\AppData\Local\Temp\ipykernel_30876\4098678742.py", line 31, in objective
    history = model.fit(
  File "C:\Users\alfre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\keras\src\utils\traceback_utils.py", line 117, in error_handler
    return fn(*args, **kwargs)
  File "C:\Users\alfre\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.9_qbz5n2kfra8p0\LocalCache\local-packages\Python39\site-packages\keras\src\backend\tensorflow\trai

KeyboardInterrupt: 